# 🐕 Hello, RL! — 用训狗的例子理解强化学习

这个 notebook 不需要你懂任何机器学习。
你只需要知道 Python 的基本语法（if/for/list）。

**目标**：用 15 分钟，理解强化学习的三个核心概念：
- **State（状态）** — 此刻发生了什么
- **Action（动作）** — 能做什么选择
- **Reward（奖励）** — 做对了加分，做错了扣分

## 1. 故事：训练一只狗学会"坐下"

想象你有一只刚养的狗。你喊"坐下"，它听不懂——它会趴下、歪头、或者不理你。

你的训练方式是：
- 狗**蹲下**了 → 给一块肉干 🦴
- 狗不理你 → 不给
- 狗扑你 → 说"不行"（负奖励）

反复几十次后，狗学会了：**听见"坐下" → 立刻蹲下 → 有肉干**。

这就是强化学习。我们现在用代码来模拟这个过程。

## 2. 最简单的版本：一步学习

狗只有一个状态（听到"坐下"），三个动作（蹲下 / 趴下 / 不理）。
我们用一个 Python 字典来记"每个动作大概能得多少分"——这就是 **Q 表** 的雏形。

In [1]:
# 这是一张"动作打分表"——最初啥都不知道，全是 0
q_table = {
    "蹲下": 0.0,
    "趴下": 0.0,
    "不理": 0.0
}

print("初始 Q 表：", q_table)

初始 Q 表： {'蹲下': 0.0, '趴下': 0.0, '不理': 0.0}


In [2]:
# 模拟一次训练：狗蹲下了 → 给奖励 +1
# Q 表更新规则（这就是 Q-learning 的核心！）：
# 新分数 = 旧分数 + 学习率 × (实际奖励 - 旧分数)

learning_rate = 0.1  # 每次学 10%（步子别太大）
action = "蹲下"
reward = 1.0         # 做对了！给 +1 分

old_score = q_table[action]
new_score = old_score + learning_rate * (reward - old_score)
q_table[action] = new_score

print(f"动作 '{action}' 得分: {old_score:.1f} → {new_score:.1f}")
print("更新后的 Q 表：", q_table)

动作 '蹲下' 得分: 0.0 → 0.1
更新后的 Q 表： {'蹲下': 0.1, '趴下': 0.0, '不理': 0.0}


### 上面这行公式是什么意思？

```
新分数 = 旧分数 + 学习率 × (实际奖励 - 旧分数)
```

- `(实际奖励 - 旧分数)` = 这次结果比预期好多少？
  - 好（+1 减 0 = +1）→ 加分
  - 差（-1 减 0.5 = -1.5）→ 扣分
- `学习率 ×` = 每次只改一点点（太高会反复横跳，太低学太慢）

这就是强化学习的本质：**用实际结果来修正预估**。

## 3. 多轮训练：反复强化正确的行为

In [3]:
import random

# 初始化 Q 表（全 0）
q_table = {"蹲下": 0.0, "趴下": 0.0, "不理": 0.0}

# 模拟"训练 50 轮"
# 规则：狗蹲下→给 +1，狗趴下→给 -1，不理→给 0
for round_num in range(1, 51):
    # 选一个动作（前 10 轮随机试，后面选最高分）
    if round_num <= 10:
        action = random.choice(["蹲下", "趴下", "不理"])
    else:
        action = max(q_table, key=q_table.get)  # 选当前最高分的动作
    
    # 环境反馈
    reward_map = {"蹲下": 1.0, "趴下": -1.0, "不理": 0.0}
    reward = reward_map[action]
    
    # Q 表学习
    learning_rate = 0.1
    q_table[action] += learning_rate * (reward - q_table[action])
    
    if round_num % 10 == 0:
        print(f"第 {round_num:2d} 轮后: {dict((k, round(v,3)) for k,v in q_table.items())}")

第 10 轮后: {'蹲下': 0.41, '趴下': -0.19, '不理': 0.0}
第 20 轮后: {'蹲下': 0.794, '趴下': -0.19, '不理': 0.0}
第 30 轮后: {'蹲下': 0.928, '趴下': -0.19, '不理': 0.0}
第 40 轮后: {'蹲下': 0.975, '趴下': -0.19, '不理': 0.0}
第 50 轮后: {'蹲下': 0.991, '趴下': -0.19, '不理': 0.0}


你看到了吗？**"蹲下"的分数逐渐趋近 +1.0，"趴下"趋近 -1.0。**

这就是强化学习在"强化"正确的行为，"弱化"错误的行为。

## 4. 探索 vs 利用 — 强化学习最重要的权衡

前 10 轮我们随机选动作（**探索**），之后选最高分动作（**利用**）。

如果从第 1 轮就只选"目前最高分"的动作：
- 假设狗第一次随机做了"不理"（得分 0）
- 而其他动作也是 0 — 算法就卡住了，永远选"不理"
- 它永远不会发现"蹲下"能得 +1！

**这就是为什么训练 G1 的时候也会有一部分随机动作——要让机器人试过各种可能性。**

## 5. 从"训狗"到"训 G1 机器人"

把上面的概念换到 G1 上：

| 训狗 | G1 训练 |
|------|---------|
| State: 听到"坐下" | State: 29 个关节的角度 + 身体倾斜角度 + 当前速度 |
| Action: 蹲下 / 趴下 / 不理 | Action: 29 个关节各自转多少度（连续值！） |
| Reward: 肉干 +1 / 不理 0 / 扑人 -1 | Reward: 走稳 +1 / 身体歪 -0.5 / 摔倒 -5 |
| Q 表: 3 行（3 个动作的手动表格） | 神经网络: 几百万个参数（自动"算"分） |

Q 表只能处理"几个动作"。G1 有 29 个关节，每个关节可以转 0° 到 180°，总共**无穷多种组合**——没法画表。

**所以我们用神经网络代替 Q 表：输入"现在是什么姿势"，输出"这样动大概得多少分"。**

下一个 notebook 就来写这个神经网络！

---

## 总结

这个 notebook 包含了强化学习的全部核心思想：

1. **State → Action → Reward** 循环
2. **Q 表** = "每个动作大概能得多少分"的记录
3. **学习规则** = 用实际奖励来修正预估（新分 = 旧分 + α × (实际 - 预估)）
4. **探索 vs 利用** = 偶尔要随机试试，不能只做"目前最赚的"

👉 下一步：[02_q_learning.ipynb](02_q_learning.ipynb) — 网格世界里的老鼠找奶酪，真正跑通 Q-learning 完整算法。